# 04 - DPR: Dense Passage Retrieval

This notebook walks you through:

1. Installing the extra dependencies needed for DPR
2. Indexing passages (encoding + storing dense vectors in PostgreSQL)
3. Running search queries with DPR
4. Evaluating retrieval quality against the `qrels` ground truth

 > **Prerequisite:** You must have run `01_data_preparation.ipynb` first so the
 > `passages`, `queries`, and `qrels` tables are populated.

In [ ]:
import sys
from pathlib import Path

# Resolve project root from notebook location
project_root = Path.cwd().resolve().parent
print(f"Project root: {project_root}")

# Ensure project root is importable so `src` package can be resolved from notebooks/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Check for .env file at project root
env_path = project_root / ".env"
if env_path.exists():
    print(f"Found .env file: {env_path}")
else:
    print("No .env found at project root. Default values from config.py will be used.")
    print("You can create one manually if needed.")

## 1) Install Python dependencies

Run this once per environment.

In [ ]:
import sys
import subprocess

requirements_file = project_root / "requirements.txt"
print(f"Installing dependencies from: {requirements_file}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
    check=True,
)

## 2) Index passages with DPR

This step encodes passages from the `passages` table and stores the resulting dense vectors in the `dpr` table.

You can re-run safely - already-indexed passages are skipped.

### Optional fast path - import precomputed DPR vectors from CSV

If you already have a full `dpr_data.csv`, you can load it directly into PostgreSQL and skip local model encoding.

This is usually much faster than re-indexing with the model.

Expected CSV format:
- `passage_id` (BIGINT)
- `embedding` (text in pgvector-compatible format, e.g. `[0.12, -0.34, ...]`)

By default this cell truncates and fully replaces local DPR vectors.

If you run this import successfully, you can skip the next `index_passages(...)` cell.

In [ ]:
import time
from src.database.connection import get_connection

csv_path = project_root / "src" / "dpr" / "dpr_data.csv"
print(f"CSV path: {csv_path}")
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

def format_duration(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    return f"{seconds / 60:.2f} min"

def log_step(step_name: str, start_time: float, global_start: float) -> None:
    step_elapsed = time.time() - start_time
    total_elapsed = time.time() - global_start
    print(
        f"[{step_name}] done in {format_duration(step_elapsed)} "
        f"(total elapsed: {format_duration(total_elapsed)})",
        flush=True,
    )

conn = get_connection()
cur = conn.cursor()

try:
    t0 = time.time()
    print("Starting DPR CSV import...", flush=True)

    # Faster bulk-load settings for this transaction only
    step_t0 = time.time()
    cur.execute("SET LOCAL synchronous_commit = OFF")
    cur.execute("SET LOCAL work_mem = '256MB'")
    cur.execute("SET LOCAL maintenance_work_mem = '1GB'")
    log_step("Session setup", step_t0, t0)

    # Staging table for very fast COPY from CSV
    step_t0 = time.time()
    cur.execute("DROP TABLE IF EXISTS dpr_import")
    cur.execute(
        """
        CREATE UNLOGGED TABLE dpr_import (
            passage_id BIGINT,
            embedding TEXT
        )
        """
    )
    log_step("Create staging table", step_t0, t0)

    # Bulk load CSV (with header)
    step_t0 = time.time()
    print("Starting COPY from CSV into staging table...", flush=True)
    with open(csv_path, "r", encoding="utf-8") as f:
        cur.copy_expert(
            """
            COPY dpr_import (passage_id, embedding)
            FROM STDIN
            WITH (FORMAT csv, HEADER true)
            """,
            f,
        )
    log_step("COPY into staging table", step_t0, t0)

    # Count imported rows and expected rows after FK-safe join
    step_t0 = time.time()
    cur.execute("SELECT COUNT(*) FROM dpr_import")
    imported_row = cur.fetchone()
    if imported_row is None:
        raise RuntimeError("Could not read row count from dpr_import")
    imported_rows = imported_row[0]

    cur.execute(
        """
        SELECT COUNT(*)
        FROM dpr_import di
        JOIN passages p ON p.id = di.passage_id
        """
    )
    expected_row = cur.fetchone()
    if expected_row is None:
        raise RuntimeError("Could not read expected row count for dpr import")
    expected_rows = expected_row[0]
    skipped_rows = imported_rows - expected_rows

    print(f"Rows copied to staging: {imported_rows}", flush=True)
    print(f"Rows matching passages: {expected_rows}", flush=True)
    if skipped_rows:
        print(f"Rows skipped because passage_id is missing: {skipped_rows}", flush=True)
    log_step("Validate staging rows", step_t0, t0)

    # Avoid maintaining ANN index during mass insert
    step_t0 = time.time()
    cur.execute("DROP INDEX IF EXISTS idx_dpr_embedding")
    log_step("Drop ivfflat index", step_t0, t0)

    # Replace DPR content from imported data. Keep only rows whose passage_id exists in passages.
    step_t0 = time.time()
    cur.execute("TRUNCATE TABLE dpr")
    log_step("Truncate dpr", step_t0, t0)

    step_t0 = time.time()
    print("Starting INSERT into dpr...", flush=True)
    cur.execute(
        """
        INSERT INTO dpr (passage_id, embedding)
        SELECT di.passage_id, di.embedding::vector
        FROM dpr_import di
        JOIN passages p ON p.id = di.passage_id
        """
    )
    log_step("Insert into dpr", step_t0, t0)

    # Rebuild ANN index once, after load
    step_t0 = time.time()
    print("Rebuilding ivfflat index on dpr.embedding...", flush=True)
    cur.execute(
        """
        CREATE INDEX idx_dpr_embedding
        ON dpr USING ivfflat (embedding vector_cosine_ops)
        """
    )
    log_step("Rebuild ivfflat index", step_t0, t0)

    # Remove staging table
    step_t0 = time.time()
    cur.execute("DROP TABLE dpr_import")
    log_step("Drop staging table", step_t0, t0)

    # Final validation before commit
    step_t0 = time.time()
    cur.execute("SELECT COUNT(*) FROM dpr")
    inserted_row = cur.fetchone()
    if inserted_row is None:
        raise RuntimeError("Could not read final row count from dpr")
    inserted = inserted_row[0]
    if inserted != expected_rows:
        raise RuntimeError(
            f"Row count mismatch: expected {expected_rows} rows in dpr, found {inserted}"
        )
    log_step("Validate final row count", step_t0, t0)

    step_t0 = time.time()
    conn.commit()
    log_step("Commit transaction", step_t0, t0)

    elapsed = time.time() - t0
    print(f"Imported rows into dpr: {inserted}", flush=True)
    print(f"Done in {format_duration(elapsed)}", flush=True)

except Exception:
    conn.rollback()
    raise
finally:
    cur.close()
    conn.close()

In [ ]:
from src.dpr.indexer import index_passages

# Large collections are best indexed in resumable chunks.
index_passages(
    limit_passages=50_000,
    batch_size=256,
    db_fetch_size=2048,
    progress_every=1,
)

### Verify indexing

In [3]:
import pandas as pd
import warnings
from src.database.connection import get_connection

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy.*")

conn = get_connection()
try:
    count = pd.read_sql_query("SELECT COUNT(*) AS indexed FROM dpr", conn)
    print(f"Indexed passages: {count['indexed'][0]}")

    print("\nSample DPR entries:")
    sample = pd.read_sql_query(
        "SELECT * FROM dpr ORDER BY passage_id LIMIT 1",
        conn
    )
    display(sample)
finally:
    conn.close()

2026-04-16 14:50:07,186 - INFO - Database connection established successfully.


Indexed passages: 600000

Sample DPR entries:


,passage_id,embedding
0,1,"[-0.5669827,-0.71997225,-0.3903634,-0.13801464..."


## 3) Search with DPR

DPR performs dense retrieval by encoding the query and ranking passages with pgvector distance.

### Run a search query

In [4]:
from src.dpr.search import search_query

query = "what is the speed of light"

search_query(query, top_k=10)

/home/tim/Documents/Projet_Big_Data_IR/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-15 19:19:10,264 - INFO - Use pytorch device_name: cpu
2026-04-15 19:19:10,265 - INFO - Load pretrained SentenceTransformer: msmarco-distilbert-base-v4
2026-04-15 19:19:10,419 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-base-v4/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-15 19:19:10,431 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/msmarco-distilbert-base-v4/b2f66c95aba1481a880479165582020c2b9b64d7/modules.json "HTTP/1.1 200 OK"


Loading model msmarco-distilbert-base-v4 (768 dim)...


2026-04-15 19:19:10,542 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-base-v4/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-15 19:19:10,553 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/msmarco-distilbert-base-v4/b2f66c95aba1481a880479165582020c2b9b64d7/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-15 19:19:10,663 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-base-v4/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-15 19:19:10,675 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/msmarco-distilbert-base-v4/b2f66c95aba1481a880479165582020c2b9b64d7/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-15 19:19:10,789 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-d

Model loaded successfully. Dimension: 768


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.02it/s]
2026-04-15 19:19:12,550 - INFO - Database connection established successfully.


Query: 'what is the speed of light'
Latency: 25.2 ms

--- TOP 10 RESULTS ---

#1 (passage_id=165548, score=0.5338)
The speed (or sometimes you might see it called velocity) of a wave, v, is how far the wave travels in a certain time. Wave speed is measured in metres per second (m/s). All the electromagnetic waves travel at 300,000,000 metres per second (3 x 10 8 m/s). 

#2 (passage_id=468319, score=0.5208)
Lightning is seen with your eyes. Because it travels at 186,000 miles per second (the speed of light) or one mile in 5.3 millionths of a second, we see it so much sooner than we hear the thunder.

#3 (passage_id=380548, score=0.5151)
Sir Isaac Newton computed the speed of sound in air as 979 feet per second (298 m/s), which is too low by about 15%, but had neglected the effect of fluctuating temperature; that was later rectified by Laplace.

#4 (passage_id=380552, score=0.5120)
The speed of sound is the distance travelled per unit time by a sound wave propagating through an elastic m

### Try your own queries

In [5]:
from src.dpr.search import search_query

for q in ["how old is the earth", "who wrote hamlet", "symptoms of diabetes"]:
    print(f"\nRunning query: {q}")
    search_query(q, top_k=3)


Running query: how old is the earth


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.18it/s]
2026-04-15 19:19:25,108 - INFO - Database connection established successfully.


Query: 'how old is the earth'
Latency: 20.9 ms

--- TOP 3 RESULTS ---

#1 (passage_id=218781, score=0.5983)
Geologic and Biological Timeline of the Earth. Astronomical and geological evidence indicates that the Universe is approximately 13,820 million years old[42], and our solar system is about 4,567 million years old. Earth's Moon formed 4,450 million years ago, just 50 million years after the Earth's formation. Geologic Time Scale-A categorization of geological events based on successively smaller time spans: eons, eras, periods, epochs, and ages. Hadean-The earliest eon in the history of the Earth from the first accretion of planetary material until the date of the oldest known rocks.

#2 (passage_id=503310, score=0.5712)
History Earth is believed to have formed about 5 billion years ago. In the first 500 million years a dense atmosphere emerged from the vapor and gases that were expelled during degassing of the planet's interior. These gases may have consisted of hydrogen (H2), wa

Batches: 100%|██████████| 1/1 [00:00<00:00, 59.18it/s]
2026-04-15 19:19:25,176 - INFO - Database connection established successfully.


Query: 'who wrote hamlet'
Latency: 18.4 ms

--- TOP 3 RESULTS ---

#1 (passage_id=427828, score=0.6031)
William Shakespeare, also known as the Bard, is responsible for some of the best plays and poetry ever written in the English language. His most well-known works include Romeo and Juliet, A Midsummer Night’s Dream, The Taming of the Shrew, Macbeth and Hamlet. 1 Richard Burbage (1567-1619), known as a method actor, played the lead roles in many of Shakespeare’s plays, Hamlet, Othello, Richard III, and King Lear, as well as those written by Ben Jonson.

#2 (passage_id=495054, score=0.5472)
Shakespeare Fun Fact. The first known English play to be written in blank verse was Gorboduc, written in 1561 by Thomas Sackville and Thomas Norton. Any English Renaissance playwright not named William Shakespeare tends to be overshadowed by the Bard's reputation.

#3 (passage_id=265174, score=0.5285)
Best Answer: Of Mice and Men is a novella written by Nobel Prize-winning author John Steinbeck. Publ

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.21it/s]
2026-04-15 19:19:25,367 - INFO - Database connection established successfully.


Query: 'symptoms of diabetes'
Latency: 23.1 ms

--- TOP 3 RESULTS ---

#1 (passage_id=190329, score=0.8845)
Individuals can experience different signs and symptoms of diabetes, and sometimes there may be no signs. Some of the signs commonly experienced include: 1  Frequent urination. 2  Excessive thirst. 3  Increased hunger. 4  Weight loss. 5  Tiredness. 6  Lack of interest and concentration. 7  A tingling sensation or numbness in the hands or feet. 

#2 (passage_id=190324, score=0.8756)
Diabetes symptoms occur when blood sugar (glucose) levels in the body become abnormally elevated. The most common symptoms of diabetes include: 1  thirst. 2  fatigue. 3  frequent. 4  increased urination. 5  blurry vision. 

#3 (passage_id=260814, score=0.7857)
Type 1: Type 1 diabetes can occur at any age, but it usually starts in people younger than 30. Symptoms are usually severe and occur rapidly. They include: 1  Increased thirst. 2  Increased urination. 3  Weight loss despite increased appetite. 4 

## Notes

- The first run downloads DPR model weights from Hugging Face.
- Indexing is idempotent: already-indexed passages are skipped.
- Search calls in this notebook use `src.dpr.search` and automatically log runs to `search_logs` and `results`.
- For large collections, tune PostgreSQL/pgvector indexing strategy to improve latency.